In [2]:
# import numpy as np
# import time
# from sklearn.datasets import make_classification, fetch_openml
# from sklearn.svm import LinearSVC
# from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# from sklearn.neural_network import MLPClassifier
# from sklearn.metrics import accuracy_score
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.neighbors import NearestNeighbors

# # ==========================================
# # CORE PIPELINE: Data Condenser
# # ==========================================
# def sv_condenser(X, y, target_ratio=0.05, max_iter=3, random_state=42):
#     np.random.seed(random_state)
#     scaler = StandardScaler()
#     X_scaled = scaler.fit_transform(X)
    
#     # Split small validation set for feedback loop
#     X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.1, stratify=y, random_state=random_state)
    
#     X_condensed, y_condensed = np.empty((0, X_train.shape[1])), np.empty((0,))
    
#     for iteration in range(max_iter):
#         print(f"\n🔄 Iteration {iteration+1}/{max_iter}")
        
#         # 1. Train SVM & Extract Support Vectors
#         svm = LinearSVC(C=1.0, max_iter=1000, random_state=random_state)
#         svm.fit(X_train, y_train)
#         sv_idx = svm.support_
#         X_sv, y_sv = X_train[sv_idx], y_train[sv_idx]
        
#         # 2. SV Augmentation via Generative Models (Boundary-Aware Interpolation)
#         nn = NearestNeighbors(n_neighbors=3, metric='euclidean').fit(X_train)
#         _, idx_neighbors = nn.kneighbors(X_sv)
#         X_aug, y_aug = [], []
#         for i, sv in enumerate(X_sv):
#             neighbors = X_train[idx_neighbors[i][1:]]  # skip self
#             for nbr in neighbors:
#                 if y_train[idx_neighbors[i][1:]][neighbors.tolist().index(nbr)] == y_sv[i]:
#                     alpha = np.random.uniform(0.2, 0.8)
#                     interp = sv + alpha * (nbr - sv)
#                     noise = np.random.normal(0, 0.05 * np.linalg.norm(nbr - sv), size=interp.shape)
#                     X_aug.append(interp + noise)
#                     y_aug.append(y_sv[i])
#         X_aug, y_aug = np.array(X_aug), np.array(y_aug)
        
#         # 3. Sensitivity-Guided Sampling (on remaining non-SV data)
#         non_sv_mask = np.ones(len(X_train), dtype=bool)
#         non_sv_mask[sv_idx] = False
#         X_rem, y_rem = X_train[non_sv_mask], y_train[non_sv_mask]
        
#         # Sensitivity score = high near margin + low local density
#         margin_dist = np.abs(X_rem @ svm.coef_.T + svm.intercept_).flatten()
#         nn_rem = NearestNeighbors(n_neighbors=5).fit(X_rem)
#         dists, _ = nn_rem.kneighbors(X_rem)
#         local_density = 1 / (dists.mean(axis=1) + 1e-6)
#         sensitivity = (1 / (margin_dist + 1e-6)) * local_density
        
#         n_sample = int(target_ratio * len(X_train))
#         top_idx = np.argsort(sensitivity)[-n_sample:]
#         X_sens, y_sens = X_rem[top_idx], y_rem[top_idx]
        
#         # 4. Merge
#         X_merge = np.vstack([X_sv, X_aug, X_sens])
#         y_merge = np.concatenate([y_sv, y_aug, y_sens])
        
#         # 5. Adversarial Margin Augmentation (AMA)
#         w = svm.coef_.flatten()
#         margin_pressure = np.clip(margin_dist, 0, 0.5)  # limit perturbation radius
#         perturb = -np.sign(w) * margin_pressure[:, None] * 0.15
#         X_ama = X_merge + perturb
#         # Clip to valid feature range
#         X_ama = np.clip(X_ama, X_merge.min(axis=0), X_merge.max(axis=0))
        
#         X_condensed, y_condensed = X_ama, y_merge
        
#         # 6. Feedback Loop Check
#         proxy = LinearSVC(C=1.0, max_iter=500)
#         proxy.fit(X_condensed, y_condensed)
#         val_acc = accuracy_score(y_val, proxy.predict(X_val))
#         print(f"   ✅ Proxy Validation Accuracy: {val_acc:.4f}")
        
#         if val_acc > 0.95 or iteration == max_iter - 1:
#             print("   🛑 Converged or max iterations reached.")
#             break
#         # If accuracy is low, slightly increase sampling ratio next iteration
#         target_ratio *= 1.2
        
#     return scaler.inverse_transform(X_condensed), y_condensed

# # ==========================================
# # VALIDATION: Downstream Model Training
# # ==========================================
# def validate_condensed_dataset(X_orig, y_orig, X_cond, y_cond, random_state=42):
#     X_tr, X_te, y_tr, y_te = train_test_split(X_orig, y_orig, test_size=0.2, stratify=y_orig, random_state=random_state)
    
#     models = {
#         "RandomForest": RandomForestClassifier(n_estimators=100, random_state=random_state),
#         "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=random_state),
#         "MLP": MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=random_state)
#     }
    
#     print("\n📊 DOWNSTREAM VALIDATION RESULTS")
#     print(f"{'Model':<20} | {'Original Acc':<12} | {'Condensed Acc':<13} | {'Data Ratio':<10} | {'Train Time (s)':<15}")
#     print("-" * 90)
    
#     for name, model in models.items():
#         # Train on original
#         t0 = time.time()
#         model.fit(X_tr, y_tr)
#         acc_orig = accuracy_score(y_te, model.predict(X_te))
#         t_orig = time.time() - t0
        
#         # Train on condensed
#         t0 = time.time()
#         model.fit(X_cond, y_cond)
#         acc_cond = accuracy_score(y_te, model.predict(X_te))
#         t_cond = time.time() - t0
        
#         ratio = len(X_cond) / len(X_tr)
#         print(f"{name:<20} | {acc_orig:.4f}      | {acc_cond:.4f}       | {ratio:.3f}x       | {t_cond:.3f}")

# # ==========================================
# # MAIN EXECUTION
# # ==========================================
# if __name__ == "__main__":
#     # 1. LOAD DATASET (Replace with your own: pd.read_csv(), etc.)
#     print("📥 Loading dataset...")
#     X, y = make_classification(n_samples=5000, n_features=20, n_informative=12, 
#                                n_redundant=4, n_clusters_per_class=2, random_state=42)
#     # OR use real data: X, y = fetch_openml('adult', version=2, return_X_y=True, as_frame=False)
    
#     print(f"   Original size: {X.shape[0]} samples, {X.shape[1]} features")
    
#     # 2. RUN PIPELINE
#     print("\n🚀 Running SV Condenser Pipeline...")
#     X_condensed, y_condensed = sv_condenser(X, y, target_ratio=0.05, max_iter=3)
    
#     print(f"\n📦 OUTPUT: Condensed Dataset")
#     print(f"   Shape: {X_condensed.shape}")
#     print(f"   Compression Ratio: {len(X)/len(X_condensed):.1f}x")
    
#     # 3. VALIDATE
#     validate_condensed_dataset(X, y, X_condensed, y_condensed)







In [4]:
import numpy as np
import time
from sklearn.datasets import make_classification, fetch_openml
from sklearn.svm import SVC, LinearSVC  # Added SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# ==========================================
# CORE PIPELINE: Data Condenser
# ==========================================
def sv_condenser(X, y, target_ratio=0.05, max_iter=3, random_state=42):
    np.random.seed(random_state)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Split small validation set for feedback loop
    X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.1, stratify=y, random_state=random_state)
    
    X_condensed, y_condensed = np.empty((0, X_train.shape[1])), np.empty((0,))
    
    for iteration in range(max_iter):
        print(f"\n🔄 Iteration {iteration+1}/{max_iter}")
        
        # 1. Train SVM & Extract Support Vectors
        # FIXED: LinearSVC doesn't have .support_, SVC(kernel='linear') does
        svm = SVC(kernel='linear', C=1.0, max_iter=1000, random_state=random_state)
        svm.fit(X_train, y_train)
        sv_idx = svm.support_
        X_sv, y_sv = X_train[sv_idx], y_train[sv_idx]
        
        # 2. SV Augmentation via Generative Models (Boundary-Aware Interpolation)
        nn = NearestNeighbors(n_neighbors=3, metric='euclidean').fit(X_train)
        _, idx_neighbors = nn.kneighbors(X_sv)
        X_aug, y_aug = [], []
        for i, sv in enumerate(X_sv):
            neighbors = X_train[idx_neighbors[i][1:]]  # skip self
            for nbr in neighbors:
                if y_train[idx_neighbors[i][1:]][neighbors.tolist().index(nbr)] == y_sv[i]:
                    alpha = np.random.uniform(0.2, 0.8)
                    interp = sv + alpha * (nbr - sv)
                    noise = np.random.normal(0, 0.05 * np.linalg.norm(nbr - sv), size=interp.shape)
                    X_aug.append(interp + noise)
                    y_aug.append(y_sv[i])
        X_aug, y_aug = np.array(X_aug), np.array(y_aug)
        
        # 3. Sensitivity-Guided Sampling (on remaining non-SV data)
        non_sv_mask = np.ones(len(X_train), dtype=bool)
        non_sv_mask[sv_idx] = False
        X_rem, y_rem = X_train[non_sv_mask], y_train[non_sv_mask]
        
        # Sensitivity score = high near margin + low local density
        margin_dist = np.abs(X_rem @ svm.coef_.T + svm.intercept_).flatten()
        nn_rem = NearestNeighbors(n_neighbors=5).fit(X_rem)
        dists, _ = nn_rem.kneighbors(X_rem)
        local_density = 1 / (dists.mean(axis=1) + 1e-6)
        sensitivity = (1 / (margin_dist + 1e-6)) * local_density
        
        n_sample = int(target_ratio * len(X_train))
        top_idx = np.argsort(sensitivity)[-n_sample:]
        X_sens, y_sens = X_rem[top_idx], y_rem[top_idx]
        
        # 4. Merge
        X_merge = np.vstack([X_sv, X_aug, X_sens])
        y_merge = np.concatenate([y_sv, y_aug, y_sens])
        
        # 5. Adversarial Margin Augmentation (AMA)
        w = svm.coef_.flatten()
        margin_pressure = np.clip(margin_dist, 0, 0.5)  # limit perturbation radius
        perturb = -np.sign(w) * margin_pressure[:, None] * 0.15
        X_ama = X_merge + perturb
        # Clip to valid feature range
        X_ama = np.clip(X_ama, X_merge.min(axis=0), X_merge.max(axis=0))
        
        X_condensed, y_condensed = X_ama, y_merge
        
        # 6. Feedback Loop Check
        proxy = LinearSVC(C=1.0, max_iter=500)
        proxy.fit(X_condensed, y_condensed)
        val_acc = accuracy_score(y_val, proxy.predict(X_val))
        print(f"   ✅ Proxy Validation Accuracy: {val_acc:.4f}")
        
        if val_acc > 0.95 or iteration == max_iter - 1:
            print("   🛑 Converged or max iterations reached.")
            break
        # If accuracy is low, slightly increase sampling ratio next iteration
        target_ratio *= 1.2
        
    return scaler.inverse_transform(X_condensed), y_condensed

# ==========================================
# VALIDATION: Downstream Model Training
# ==========================================
def validate_condensed_dataset(X_orig, y_orig, X_cond, y_cond, random_state=42):
    X_tr, X_te, y_tr, y_te = train_test_split(X_orig, y_orig, test_size=0.2, stratify=y_orig, random_state=random_state)
    
    models = {
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=random_state),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=random_state),
        "MLP": MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=random_state)
    }
    
    print("\n📊 DOWNSTREAM VALIDATION RESULTS")
    print(f"{'Model':<20} | {'Original Acc':<12} | {'Condensed Acc':<13} | {'Data Ratio':<10} | {'Train Time (s)':<15}")
    print("-" * 90)
    
    for name, model in models.items():
        # Train on original
        t0 = time.time()
        model.fit(X_tr, y_tr)
        acc_orig = accuracy_score(y_te, model.predict(X_te))
        t_orig = time.time() - t0
        
        # Train on condensed
        t0 = time.time()
        model.fit(X_cond, y_cond)
        acc_cond = accuracy_score(y_te, model.predict(X_te))
        t_cond = time.time() - t0
        
        ratio = len(X_cond) / len(X_tr)
        print(f"{name:<20} | {acc_orig:.4f}      | {acc_cond:.4f}       | {ratio:.3f}x       | {t_cond:.3f}")

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    # 1. LOAD DATASET (Replace with your own: pd.read_csv(), etc.)
    print("📥 Loading dataset...")
    X, y = make_classification(n_samples=5000, n_features=20, n_informative=12, 
                               n_redundant=4, n_clusters_per_class=2, random_state=42)
    # OR use real  X, y = fetch_openml('adult', version=2, return_X_y=True, as_frame=False)
    
    print(f"   Original size: {X.shape[0]} samples, {X.shape[1]} features")
    
    # 2. RUN PIPELINE
    print("\n🚀 Running SV Condenser Pipeline...")
    X_condensed, y_condensed = sv_condenser(X, y, target_ratio=0.05, max_iter=3)
    
    print(f"\n📦 OUTPUT: Condensed Dataset")
    print(f"   Shape: {X_condensed.shape}")
    print(f"   Compression Ratio: {len(X)/len(X_condensed):.1f}x")
    
    # 3. VALIDATE
    validate_condensed_dataset(X, y, X_condensed, y_condensed)

📥 Loading dataset...
   Original size: 5000 samples, 20 features

🚀 Running SV Condenser Pipeline...

🔄 Iteration 1/3


/home/arif/.local/share/pipx/venvs/jupyterlab/lib/python3.12/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


ValueError: operands could not be broadcast together with shapes (2133,20) (3759,20) 

In [5]:
import numpy as np
import time
from sklearn.datasets import make_classification, fetch_openml
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# ==========================================
# CORE PIPELINE: Data Condenser
# ==========================================
def sv_condenser(X, y, target_ratio=0.05, max_iter=3, random_state=42):
    np.random.seed(random_state)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Split small validation set for feedback loop
    X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.1, stratify=y, random_state=random_state)
    
    X_condensed, y_condensed = np.empty((0, X_train.shape[1])), np.empty((0,))
    
    for iteration in range(max_iter):
        print(f"\n🔄 Iteration {iteration+1}/{max_iter}")
        
        # 1. Train SVM & Extract Support Vectors
        svm = SVC(kernel='linear', C=1.0, max_iter=1000, random_state=random_state)
        svm.fit(X_train, y_train)
        sv_idx = svm.support_
        X_sv, y_sv = X_train[sv_idx], y_train[sv_idx]
        
        # 2. SV Augmentation via Generative Models (Boundary-Aware Interpolation)
        nn = NearestNeighbors(n_neighbors=3, metric='euclidean').fit(X_train)
        _, idx_neighbors = nn.kneighbors(X_sv)
        X_aug, y_aug = [], []
        for i, sv in enumerate(X_sv):
            neighbor_indices = idx_neighbors[i][1:]  # skip self
            # FIXED: Safe index-based iteration replaces broken .index() lookup
            for nbr_idx in neighbor_indices:
                if y_train[nbr_idx] == y_sv[i]:
                    nbr = X_train[nbr_idx]
                    alpha = np.random.uniform(0.2, 0.8)
                    interp = sv + alpha * (nbr - sv)
                    noise = np.random.normal(0, 0.05 * np.linalg.norm(nbr - sv), size=interp.shape)
                    X_aug.append(interp + noise)
                    y_aug.append(y_sv[i])
        
        # Handle case where no neighbors match (prevents empty array shape mismatch)
        if len(X_aug) == 0:
            X_aug = np.empty((0, X_train.shape[1]))
            y_aug = np.empty((0,))
        else:
            X_aug, y_aug = np.array(X_aug), np.array(y_aug)
        
        # 3. Sensitivity-Guided Sampling (on remaining non-SV data)
        non_sv_mask = np.ones(len(X_train), dtype=bool)
        non_sv_mask[sv_idx] = False
        X_rem, y_rem = X_train[non_sv_mask], y_train[non_sv_mask]
        
        # Sensitivity score = high near margin + low local density
        margin_dist = np.abs(X_rem @ svm.coef_.T + svm.intercept_).flatten()
        nn_rem = NearestNeighbors(n_neighbors=5).fit(X_rem)
        dists, _ = nn_rem.kneighbors(X_rem)
        local_density = 1 / (dists.mean(axis=1) + 1e-6)
        sensitivity = (1 / (margin_dist + 1e-6)) * local_density
        
        n_sample = int(target_ratio * len(X_train))
        top_idx = np.argsort(sensitivity)[-n_sample:]
        X_sens, y_sens = X_rem[top_idx], y_rem[top_idx]
        
        # 4. Merge
        X_merge = np.vstack([X_sv, X_aug, X_sens])
        y_merge = np.concatenate([y_sv, y_aug, y_sens])
        
        # 5. Adversarial Margin Augmentation (AMA)
        w = svm.coef_.flatten()
        margin_pressure = np.clip(margin_dist, 0, 0.5)  # limit perturbation radius
        perturb = -np.sign(w) * margin_pressure[:, None] * 0.15
        X_ama = X_merge + perturb
        # Clip to valid feature range
        X_ama = np.clip(X_ama, X_merge.min(axis=0), X_merge.max(axis=0))
        
        X_condensed, y_condensed = X_ama, y_merge
        
        # 6. Feedback Loop Check
        proxy = LinearSVC(C=1.0, max_iter=500)
        proxy.fit(X_condensed, y_condensed)
        val_acc = accuracy_score(y_val, proxy.predict(X_val))
        print(f"   ✅ Proxy Validation Accuracy: {val_acc:.4f}")
        
        if val_acc > 0.95 or iteration == max_iter - 1:
            print("   🛑 Converged or max iterations reached.")
            break
        # If accuracy is low, slightly increase sampling ratio next iteration
        target_ratio *= 1.2
        
    return scaler.inverse_transform(X_condensed), y_condensed

# ==========================================
# VALIDATION: Downstream Model Training
# ==========================================
def validate_condensed_dataset(X_orig, y_orig, X_cond, y_cond, random_state=42):
    X_tr, X_te, y_tr, y_te = train_test_split(X_orig, y_orig, test_size=0.2, stratify=y_orig, random_state=random_state)
    
    models = {
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=random_state),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=random_state),
        "MLP": MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=random_state)
    }
    
    print("\n📊 DOWNSTREAM VALIDATION RESULTS")
    print(f"{'Model':<20} | {'Original Acc':<12} | {'Condensed Acc':<13} | {'Data Ratio':<10} | {'Train Time (s)':<15}")
    print("-" * 90)
    
    for name, model in models.items():
        # Train on original
        t0 = time.time()
        model.fit(X_tr, y_tr)
        acc_orig = accuracy_score(y_te, model.predict(X_te))
        t_orig = time.time() - t0
        
        # Train on condensed
        t0 = time.time()
        model.fit(X_cond, y_cond)
        acc_cond = accuracy_score(y_te, model.predict(X_te))
        t_cond = time.time() - t0
        
        ratio = len(X_cond) / len(X_tr)
        print(f"{name:<20} | {acc_orig:.4f}      | {acc_cond:.4f}       | {ratio:.3f}x       | {t_cond:.3f}")

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    # 1. LOAD DATASET (Replace with your own: pd.read_csv(), etc.)
    print("📥 Loading dataset...")
    X, y = make_classification(n_samples=5000, n_features=20, n_informative=12, 
                               n_redundant=4, n_clusters_per_class=2, random_state=42)
    
    print(f"   Original size: {X.shape[0]} samples, {X.shape[1]} features")
    
    # 2. RUN PIPELINE
    print("\n🚀 Running SV Condenser Pipeline...")
    X_condensed, y_condensed = sv_condenser(X, y, target_ratio=0.05, max_iter=3)
    
    print(f"\n📦 OUTPUT: Condensed Dataset")
    print(f"   Shape: {X_condensed.shape}")
    print(f"   Compression Ratio: {len(X)/len(X_condensed):.1f}x")
    
    # 3. VALIDATE
    validate_condensed_dataset(X, y, X_condensed, y_condensed)

📥 Loading dataset...
   Original size: 5000 samples, 20 features

🚀 Running SV Condenser Pipeline...

🔄 Iteration 1/3


/home/arif/.local/share/pipx/venvs/jupyterlab/lib/python3.12/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


ValueError: operands could not be broadcast together with shapes (2133,20) (3759,20) 

In [6]:
import numpy as np
import time
from sklearn.datasets import make_classification, fetch_openml
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# ==========================================
# CORE PIPELINE: Data Condenser
# ==========================================
def sv_condenser(X, y, target_ratio=0.05, max_iter=3, random_state=42):
    np.random.seed(random_state)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Split small validation set for feedback loop
    X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.1, stratify=y, random_state=random_state)
    
    X_condensed, y_condensed = np.empty((0, X_train.shape[1])), np.empty((0,))
    
    for iteration in range(max_iter):
        print(f"\n🔄 Iteration {iteration+1}/{max_iter}")
        
        # 1. Train SVM & Extract Support Vectors
        svm = SVC(kernel='linear', C=1.0, max_iter=1000, random_state=random_state)
        svm.fit(X_train, y_train)
        sv_idx = svm.support_
        X_sv, y_sv = X_train[sv_idx], y_train[sv_idx]
        
        # 2. SV Augmentation via Generative Models (Boundary-Aware Interpolation)
        nn = NearestNeighbors(n_neighbors=3, metric='euclidean').fit(X_train)
        _, idx_neighbors = nn.kneighbors(X_sv)
        X_aug, y_aug = [], []
        for i, sv in enumerate(X_sv):
            neighbor_indices = idx_neighbors[i][1:]  # skip self
            for nbr_idx in neighbor_indices:
                if y_train[nbr_idx] == y_sv[i]:
                    nbr = X_train[nbr_idx]
                    alpha = np.random.uniform(0.2, 0.8)
                    interp = sv + alpha * (nbr - sv)
                    noise = np.random.normal(0, 0.05 * np.linalg.norm(nbr - sv), size=interp.shape)
                    X_aug.append(interp + noise)
                    y_aug.append(y_sv[i])
        
        if len(X_aug) == 0:
            X_aug = np.empty((0, X_train.shape[1]))
            y_aug = np.empty((0,))
        else:
            X_aug, y_aug = np.array(X_aug), np.array(y_aug)
        
        # 3. Sensitivity-Guided Sampling (on remaining non-SV data)
        non_sv_mask = np.ones(len(X_train), dtype=bool)
        non_sv_mask[sv_idx] = False
        X_rem, y_rem = X_train[non_sv_mask], y_train[non_sv_mask]
        
        margin_dist_rem = np.abs(X_rem @ svm.coef_.T + svm.intercept_).flatten()
        nn_rem = NearestNeighbors(n_neighbors=5).fit(X_rem)
        dists, _ = nn_rem.kneighbors(X_rem)
        local_density = 1 / (dists.mean(axis=1) + 1e-6)
        sensitivity = (1 / (margin_dist_rem + 1e-6)) * local_density
        
        n_sample = int(target_ratio * len(X_train))
        top_idx = np.argsort(sensitivity)[-n_sample:]
        X_sens, y_sens = X_rem[top_idx], y_rem[top_idx]
        
        # 4. Merge
        X_merge = np.vstack([X_sv, X_aug, X_sens])
        y_merge = np.concatenate([y_sv, y_aug, y_sens])
        
        # 5. Adversarial Margin Augmentation (AMA)
        w = svm.coef_.flatten()
        # FIXED: Compute margin distance for X_merge (not X_rem) to match shapes
        merge_margin_dist = np.abs(X_merge @ svm.coef_.T + svm.intercept_).flatten()
        margin_pressure = np.clip(merge_margin_dist, 0, 0.5)
        perturb = -np.sign(w) * margin_pressure[:, None] * 0.15
        X_ama = X_merge + perturb
        # Clip to valid feature range
        X_ama = np.clip(X_ama, X_merge.min(axis=0), X_merge.max(axis=0))
        
        X_condensed, y_condensed = X_ama, y_merge
        
        # 6. Feedback Loop Check
        proxy = LinearSVC(C=1.0, max_iter=500)
        proxy.fit(X_condensed, y_condensed)
        val_acc = accuracy_score(y_val, proxy.predict(X_val))
        print(f"   ✅ Proxy Validation Accuracy: {val_acc:.4f}")
        
        if val_acc > 0.95 or iteration == max_iter - 1:
            print("   🛑 Converged or max iterations reached.")
            break
        target_ratio *= 1.2
        
    return scaler.inverse_transform(X_condensed), y_condensed

# ==========================================
# VALIDATION: Downstream Model Training
# ==========================================
def validate_condensed_dataset(X_orig, y_orig, X_cond, y_cond, random_state=42):
    X_tr, X_te, y_tr, y_te = train_test_split(X_orig, y_orig, test_size=0.2, stratify=y_orig, random_state=random_state)
    
    models = {
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=random_state),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=random_state),
        "MLP": MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=random_state)
    }
    
    print("\n📊 DOWNSTREAM VALIDATION RESULTS")
    print(f"{'Model':<20} | {'Original Acc':<12} | {'Condensed Acc':<13} | {'Data Ratio':<10} | {'Train Time (s)':<15}")
    print("-" * 90)
    
    for name, model in models.items():
        t0 = time.time()
        model.fit(X_tr, y_tr)
        acc_orig = accuracy_score(y_te, model.predict(X_te))
        t_orig = time.time() - t0
        
        t0 = time.time()
        model.fit(X_cond, y_cond)
        acc_cond = accuracy_score(y_te, model.predict(X_te))
        t_cond = time.time() - t0
        
        ratio = len(X_cond) / len(X_tr)
        print(f"{name:<20} | {acc_orig:.4f}      | {acc_cond:.4f}       | {ratio:.3f}x       | {t_cond:.3f}")

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    print("📥 Loading dataset...")
    X, y = make_classification(n_samples=5000, n_features=20, n_informative=12, 
                               n_redundant=4, n_clusters_per_class=2, random_state=42)
    
    print(f"   Original size: {X.shape[0]} samples, {X.shape[1]} features")
    
    print("\n🚀 Running SV Condenser Pipeline...")
    X_condensed, y_condensed = sv_condenser(X, y, target_ratio=0.05, max_iter=3)
    
    print(f"\n📦 OUTPUT: Condensed Dataset")
    print(f"   Shape: {X_condensed.shape}")
    print(f"   Compression Ratio: {len(X)/len(X_condensed):.1f}x")
    
    validate_condensed_dataset(X, y, X_condensed, y_condensed)

📥 Loading dataset...
   Original size: 5000 samples, 20 features

🚀 Running SV Condenser Pipeline...

🔄 Iteration 1/3
   ✅ Proxy Validation Accuracy: 0.8260

🔄 Iteration 2/3


/home/arif/.local/share/pipx/venvs/jupyterlab/lib/python3.12/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/home/arif/.local/share/pipx/venvs/jupyterlab/lib/python3.12/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


   ✅ Proxy Validation Accuracy: 0.8260

🔄 Iteration 3/3
   ✅ Proxy Validation Accuracy: 0.8240
   🛑 Converged or max iterations reached.

📦 OUTPUT: Condensed Dataset
   Shape: (2232, 20)
   Compression Ratio: 2.2x

📊 DOWNSTREAM VALIDATION RESULTS
Model                | Original Acc | Condensed Acc | Data Ratio | Train Time (s) 
------------------------------------------------------------------------------------------


/home/arif/.local/share/pipx/venvs/jupyterlab/lib/python3.12/site-packages/sklearn/svm/_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


RandomForest         | 0.9300      | 0.8680       | 0.558x       | 0.593
GradientBoosting     | 0.9060      | 0.8420       | 0.558x       | 1.071
MLP                  | 0.9640      | 0.9310       | 0.558x       | 1.214
